# 🏥 Cardiovascular Risk Classifier Model Training (Google Colab)

This Jupyter Notebook was custom-developed for **Google Colab** to train and serialize the clinical predictive models used in the **Clini-SHAP Intelligent CDSS (Clinical Decision Support Suite)**.

### 📋 Dataset Context: Cleveland Clinical Heart Disease Database
### 🤖 Model Classifier Architecture: `XGBoost Classifier`

---

## ⚙️ Step 1: Environment Setup & Installations
We install the correct medical model dependencies including `xgboost`, `scikit-learn`, `shap`, `joblib`, `matplotlib`, and `seaborn` directly in Google Colab.


In [ ]:
# Install libraries in Google Colab
!pip install pandas numpy scikit-learn shap joblib matplotlib seaborn xgboost fpdf2


## 🧪 Step 2: Import Core Libraries
Load libraries required for numerical computation, exploratory data analysis (EDA), scaling pipeline, and SHAP interpretability.


In [ ]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, classification_report
import shap
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier

# Setup visualization parameters
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12


## 📂 Step 3: Load Clinical Dataset
We fetch the dataset from the live repository and load it into a pandas DataFrame.

Loads the Cleveland Heart Disease Dataset featuring patients classified by structural coronary blocks.


In [ ]:
# Load Cleveland Heart Disease Dataset
dataset_url = 'https://raw.githubusercontent.com/anushka06onu/AI-Healthcare-Analytics-Platform/main/data/heart.csv'
try:
    df = pd.read_csv(dataset_url)
    print('Successfully downloaded heart dataset!')
except Exception as e:
    print('Error loading:')
    raise e

print(f'Shape of Heart Disease Cohort: {df.shape}')
df.head()


## 🔬 Step 4: Train-Test Split and Clinical Preprocessing
We partition our EMR metrics into training and test cohorts (80% training, 20% test stratified) and scale features using `StandardScaler`.


In [ ]:
# Split features and outcome
X = df.drop(columns=['target'])
y = df['target']

# Split train/test cohorts
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Normalize via standard scaler
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

print('Cardio prepreprocessing succeeded!')


## 🤖 Step 5: Train EMR Predictive Classifier
We train the high-fidelity `XGBoost Classifier` model on our preprocessed training cohort.


In [ ]:
# Train cardio XGBoost classifier
model = xgb.XGBClassifier(random_state=42, eval_metric='logloss')
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

print('--- CLINICAL DIAGNOSTIC METRICS ---')
print(f'Accuracy Score:  {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision Score: {precision_score(y_test, y_pred):.4f}')
print(f'Recall Score:    {recall_score(y_test, y_pred):.4f}')
print(f'ROC-AUC Score:   {roc_auc_score(y_test, y_pred_proba):.4f}')


## 🔍 Step 6: Local and Global SHAP Attributions
To ensure 100% medical interpretability and accountabilities, we initialize SHAP explainer objects and pre-calculate SHAP matrices.


In [ ]:
print('Pre-computing heart disease SHAP attributions...')
bg_sample = shap.sample(X_train_scaled, min(30, len(X_train_scaled)))
test_sample = shap.sample(X_test_scaled, min(100, len(X_test_scaled)))

explainer = shap.KernelExplainer(model.predict_proba, bg_sample)
shap_values = explainer.shap_values(test_sample)

shap_matrix = shap_values[1] if isinstance(shap_values, list) else shap_values

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_matrix, test_sample, show=False)
plt.title('Cardiovascular Disease Attributions (SHAP Beeswarm Plot)', fontsize=14)
plt.tight_layout()
plt.show()


## 💾 Step 7: Export Serialized Components
We export the trained model classifier, the calibrated scaler, baseline training feature names, and pre-computed SHAP files for seamless integration into the CDSS application.


In [ ]:
os.makedirs('models', exist_ok=True)
os.makedirs('shap_files', exist_ok=True)

joblib.dump(model, 'models/heart_model.pkl')
joblib.dump(scaler, 'models/heart_scaler.pkl')
joblib.dump(list(scaler.feature_names_in_), 'models/heart_columns.joblib')
joblib.dump(X_train, 'models/heart_X_train.joblib')

joblib.dump(None, 'shap_files/heart_explainer.joblib')
joblib.dump(shap_matrix, 'shap_files/heart_shap_values.joblib')
joblib.dump(test_sample, 'shap_files/heart_X_test.joblib')

print('Cardio EMR serializations successfully completed!')
